# HUẤN LUYỆN M1 (CodeBERT) TRÊN BIGCLONEBENCH JAVA (GPU)

Notebook này thực hiện huấn luyện mô hình **M1 (CodeBERT)** trên BigCloneBench Java sử dụng GPU Kaggle.

### Thông tin mô hình:
- **Mã mô hình:** M1
- **Tên phương pháp:** CodeBERT
- **File cấu hình:** `configs/exp002_codebert_bcb.yaml`
- **Dataset liên kết:** `semanticcodeclonedetection-dataset-m4`
- **Đầu ra tại `/kaggle/working/`:** `best_checkpoint.pt`, `training_summary.json`, `test_metrics.json`

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ
import base64, io, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

def sh(cmd, **kw):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.stdout:
        print(r.stdout.strip())
    if r.stderr:
        print("[stderr]", r.stderr.strip())
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={r.returncode}): {cmd}")
    return r.stdout

print("Python:", sys.version.split()[0])

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN MỚI NHẤT (SELF-CONTAINED)
WORK_DIR = Path("/kaggle/working/project")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = "..."

print(f"Giải nén mã nguồn ({len(BUNDLE_B64)/1024:.1f} KB) vào {WORK_DIR}...")
buf = io.BytesIO(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=buf, mode="r:gz") as tar:
    tar.extractall(WORK_DIR)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
sh("ls -la configs/exp002_codebert_bcb.yaml")

In [ ]:
# 3. CÀI ĐẶT THƯ VIỆN & XÁC THỰC GPU
sh("pip install -q 'tree-sitter>=0.23.0' 'tree-sitter-java>=0.23.5' 'tree-sitter-cpp>=0.23.0' 'datasets>=2.14.0' 'transformers>=4.30.0' 'pyarrow' 'tqdm' 'pyyaml' 'scikit-learn'")

import torch
assert torch.cuda.is_available(), "Cần bật GPU trong Settings -> Accelerator -> GPU (T4/P100)!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"GPU SẴN SÀNG: {gpu_name} | VRAM: {vram_gb} GB")
sh("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv")

In [ ]:
# 4. LIÊN KẾT ĐẶC TRƯNG TỪ DATASET NẾU CẦN (AST / CPG)
in_dir = Path("/kaggle/input/semanticcodeclonedetection-dataset-m4")
if in_dir.exists():
    print(f"Dataset đính kèm: {in_dir}")
    sh(f"ls -lh {in_dir}")
    # Nạp AST cache nếu là M3
    ast_p = in_dir / "bcb_ast_cache.npz"
    if ast_p.exists():
        os.makedirs("artifacts/bcb_features", exist_ok=True)
        sh(f"cp {ast_p} artifacts/bcb_features/")
    # Nạp CPG archive nếu là M4
    cpg_tar = in_dir / "bcb_cpg_export.tar.gz"
    if cpg_tar.exists():
        os.makedirs("artifacts", exist_ok=True)
        print("Đang giải nén CPG cache...")
        sh(f"tar -xzf {cpg_tar} -C artifacts/")
else:
    print("Chưa phát hiện dataset đính kèm, sử dụng đặc trưng mặc định.")

In [ ]:
# 5. TẢI CÁC SPLIT DỮ LIỆU ĐỂ HUẤN LUYỆN (TRAIN 50K, VAL 5K, TEST 5K)
data_raw = Path("data/raw/bigclonebench")
data_raw.mkdir(parents=True, exist_ok=True)
print("Tải các phân tập dữ liệu BigCloneBench (Train: 50k, Val: 5k, Test: 5k)...")
sh("python scripts/download_bcb_subsets.py --split train --max-samples 50000")
sh("python scripts/download_bcb_subsets.py --split validation --max-samples 5000")
sh("python scripts/download_bcb_subsets.py --split test --max-samples 5000")
sh("ls -lh data/raw/bigclonebench/")

In [ ]:
# 6. PREFLIGHT & DIAGNOSTIC CHECK
print("--- PREFLIGHT M1 ---")
sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode preflight --device cuda")
print("\n--- DIAGNOSTIC CHECK M1 ---")
sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode diagnostic --device cuda")

In [ ]:
# 7. TIẾN HÀNH HUẤN LUYỆN M1 (CodeBERT) TRÊN GPU
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN CHÍNH THỨC M1 (CodeBERT)...")
print("=" * 60)
TIME_BUDGET_S = 9 * 3600
sh(f"python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode train --device cuda --time-budget-s {TIME_BUDGET_S}")

In [ ]:
# 8. ĐÁNH GIÁ MÔ HÌNH M1 TRÊN TẬP TEST
print("=" * 60)
print("ĐÁNH GIÁ MÔ HÌNH M1 TRÊN TẬP TEST...")
print("=" * 60)
sh("python -m src.experiments.bcb_runner --config configs/exp002_codebert_bcb.yaml --mode evaluate --device cuda")

In [ ]:
# 9. ĐÓNG GÓI KẾT QUẢ VÀO /kaggle/working/
exp_id = json.load(open(f"{WORK_DIR}/{config_file}"))["experiment"]["id"].lower().replace("-", "_")
res_dir = Path("results") / exp_id
dest_dir = Path("/kaggle/working/m1_results")
dest_dir.mkdir(parents=True, exist_ok=True)

if res_dir.exists():
    for f in res_dir.glob("*"):
        sh(f"cp -r {f} {dest_dir}/")
        if f.suffix in (".json", ".pt"):
            sh(f"cp {f} /kaggle/working/")

TAR_RESULTS = Path("/kaggle/working/m1_results.tar.gz")
sh(f"tar -czf {TAR_RESULTS} results/")
sh("ls -lh /kaggle/working/*.json /kaggle/working/*.tar.gz 2>/dev/null || ls -lh /kaggle/working/")

if (res_dir / "test_metrics.json").exists():
    metrics = json.loads((res_dir / "test_metrics.json").read_text(encoding="utf-8"))
    print("\nBẢNG TEST METRICS CỦA M1:\n", json.dumps(metrics, indent=2))
print("\nHUẤN LUYỆN VÀ ĐÁNH GIÁ M1 HOÀN TẤT THÀNH CÔNG!")